In [1]:
import os
import json
import csv
import time
import heapq
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import binary_dilation
from collections import deque
from tqdm import tqdm
import platform

# Фиксация для воспроизводимости
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

# 1. КОНФИГУРАЦИЯ
CONFIG = {
    "grid_size": 30,
    "patch_size": 30,
    "densities": [0.10, 0.20, 0.30, 0.40, 0.50],
    "maps_per_density": 100,  
    "seed": 42,
    "device": "cpu",
    "cnn_epochs": 10,
    "warmup_runs": 3,
    "measurement_runs": 3,
}

def get_env_info():
    return {
        "os": f"{platform.system()} {platform.release()}",
        "processor": platform.processor(),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "note": "Single-threaded CPU mode for reproducibility."
    }


# 2. ГЕНЕРАЦИЯ И ПРАВИЛА ДВИЖЕНИЯ
def generate_map(size, density, rng):
    noise = rng.random((size, size)) < (density * 0.5)
    seeds = rng.random((size, size)) < (density * 0.1)
    clusters = binary_dilation(seeds, iterations=3)
    grid = (noise | clusters).astype(np.uint8)
    grid[0, 0] = 0
    grid[size-1, size-1] = 0
    return grid

def get_neighbors_8way_strict(grid, current):
    x, y = current
    size = grid.shape[0]
    neighbors = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                if dx != 0 and dy != 0:
                    if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                neighbors.append(((nx, ny), cost))
    return neighbors

def is_reachable_strict(grid, start, goal):
    size = grid.shape[0]
    visited = np.zeros((size, size), dtype=bool)
    queue = deque([start])
    visited[start] = True
    while queue:
        x, y = queue.popleft()
        if (x, y) == goal: return True
        for neighbor, _ in get_neighbors_8way_strict(grid, (x, y)):
            if not visited[neighbor]:
                visited[neighbor] = True
                queue.append(neighbor)
    return False

def compute_dijkstra_optimal(grid, start, goal):
    size = grid.shape[0]
    open_set = [(0, start)]
    g_score = {start: 0}
    came_from = {}
    while open_set:
        g, current = heapq.heappop(open_set)
        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return True, g, list(reversed(path))
        for neighbor, cost in get_neighbors_8way_strict(grid, current):
            tentative_g = g_score[current] + cost
            if tentative_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                heapq.heappush(open_set, (tentative_g, neighbor))
    return False, -1, []

def validate_path(grid, path, start, goal):
    if not path or path[0] != start or path[-1] != goal: return False
    for i in range(len(path) - 1):
        x1, y1 = path[i]
        x2, y2 = path[i+1]
        if grid[x2, y2] == 1: return False
        if abs(x1 - x2) > 1 or abs(y1 - y2) > 1: return False
        if abs(x1 - x2) == 1 and abs(y1 - y2) == 1:
            if grid[x1+ (x2-x1), y1] == 1 or grid[x1, y1+ (y2-y1)] == 1: return False
    return True

# 3. CNN И DATASET
class HeuristicDataset(Dataset):
    def __init__(self, maps, h_stars, patch_size):
        self.samples = []
        pad = patch_size // 2
        for grid, h_star in zip(maps, h_stars):
            size = grid.shape[0]
            goal = (size-1, size-1)
            y_coords, x_coords = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
            max_dist = np.sqrt(2 * (patch_size**2))
            dist_to_goal = np.clip(np.sqrt((x_coords - goal[1])**2 + (y_coords - goal[0])**2) / max_dist, 0.0, 1.0)
            
            padded_grid = np.pad(grid, pad, mode='constant', constant_values=1)
            padded_dist = np.pad(dist_to_goal, pad, mode='constant', constant_values=1.0)
            
            for x in range(size):
                for y in range(size):
                    if grid[x, y] == 0 and h_star[x, y] != np.inf:
                        patch_obs = padded_grid[x : x+patch_size, y : y+patch_size]
                        patch_dist = padded_dist[x : x+patch_size, y : y+patch_size]
                        self.samples.append((np.stack([patch_obs, patch_dist], axis=0), h_star[x, y]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        return self.fc(torch.flatten(x, 1))


# 4. ПРОФИЛИРОВАННЫЙ ПОИСК
def astar_profiled(grid, start, goal, cnn_model=None, patch_size=30):
    size = grid.shape[0]
    open_set = [(0, 0, 0, start)]
    g_score = {start: 0}
    came_from = {}
    counter = 1
    h_cache = {} if cnn_model is not None else None
    
    t_prep_maps_start = time.perf_counter()
    if cnn_model is not None:
        pad = patch_size // 2
        padded_grid = np.pad(grid, pad, mode='constant', constant_values=1)
        y_c, x_c = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
        max_dist = np.sqrt(2 * (patch_size**2))
        dist_map = np.clip(np.sqrt((x_c - goal[1])**2 + (y_c - goal[0])**2) / max_dist, 0.0, 1.0)
        padded_dist = np.pad(dist_map, pad, mode='constant', constant_values=1.0)
    t_prep_maps_ms = (time.perf_counter() - t_prep_maps_start) * 1000
    
    n_expanded = 0
    n_cnn_calls = 0
    n_cache_hits = 0
    t_prepare_ms = t_prep_maps_ms
    t_infer_ms = 0.0
    t_queue_ms = 0.0
    
    t_total_start = time.perf_counter()
    
    while open_set:
        t_q_start = time.perf_counter()
        f, g, _, current = heapq.heappop(open_set)
        t_queue_ms += (time.perf_counter() - t_q_start) * 1000
        
        if current == goal: break
        if g > g_score.get(current, float('inf')): continue
            
        n_expanded += 1
        
        for neighbor, cost in get_neighbors_8way_strict(grid, current):
            tentative_g = g_score[current] + cost
            if tentative_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                
                if cnn_model is not None:
                    if neighbor in h_cache:
                        h_cnn = h_cache[neighbor]
                        n_cache_hits += 1
                    else:
                        t_p_start = time.perf_counter()
                        px, py = neighbor
                        patch_obs = padded_grid[px : px+patch_size, py : py+patch_size]
                        patch_dist = padded_dist[px : px+patch_size, py : py+patch_size]
                        tensor_in = torch.tensor(np.stack([patch_obs, patch_dist], axis=0), dtype=torch.float32).unsqueeze(0)
                        t_prepare_ms += (time.perf_counter() - t_p_start) * 1000
                        
                        t_i_start = time.perf_counter()
                        with torch.no_grad():
                            h_cnn = cnn_model(tensor_in).item()
                        t_infer_ms += (time.perf_counter() - t_i_start) * 1000
                        n_cnn_calls += 1
                        h_cache[neighbor] = h_cnn
                    
                    h_eucl = np.sqrt((neighbor[0] - goal[0])**2 + (neighbor[1] - goal[1])**2)
                    h_used = max(h_cnn, h_eucl)
                else:
                    h_used = np.sqrt((neighbor[0] - goal[0])**2 + (neighbor[1] - goal[1])**2)
                
                f_score = tentative_g + h_used
                t_q_start = time.perf_counter()
                heapq.heappush(open_set, (f_score, tentative_g, counter, neighbor))
                t_queue_ms += (time.perf_counter() - t_q_start) * 1000
                counter += 1
                
    t_path_start = time.perf_counter()
    if current == goal:
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        path = list(reversed(path))
        path_cost = g_score[goal]
        success = True
    else:
        path = []
        path_cost = -1
        success = False
    t_path_ms = (time.perf_counter() - t_path_start) * 1000
    
    return {
        "t_total_ms": (time.perf_counter() - t_total_start) * 1000,
        "t_prepare_ms": t_prepare_ms,
        "t_infer_ms": t_infer_ms,
        "t_queue_ms": t_queue_ms,
        "t_path_ms": t_path_ms,
        "n_expanded": n_expanded,
        "n_cnn_calls": n_cnn_calls,
        "n_cache_hits": n_cache_hits,
        "path_cost": path_cost,
        "success": success,
        "path": path
    }

# 5. ГЛАВНЫЙ ЦИКЛ
def run_experiment():
    print(" ЗАПУСК ВОСПРОИЗВОДИМОГО ЭКСПЕРИМЕНТА ")
    os.makedirs("output_data", exist_ok=True)
    
    rng = np.random.default_rng(CONFIG["seed"])
    start, goal = (0,0), (CONFIG["grid_size"]-1, CONFIG["grid_size"]-1)
    
    all_train_maps, all_train_h_stars = [], []
    test_pools = {d: [] for d in CONFIG["densities"]}
    generation_stats = {d: {"attempts": 0, "reachable": 0} for d in CONFIG["densities"]}
    
    print("\n[1/4] Генерация карт...")
    for d in CONFIG["densities"]:
        count = 0
        attempts = 0
        while count < CONFIG["maps_per_density"] and attempts < CONFIG["maps_per_density"] * 20:
            grid = generate_map(CONFIG["grid_size"], d, rng)
            attempts += 1
            if is_reachable_strict(grid, start, goal):
                s_opt, c_opt, path_opt = compute_dijkstra_optimal(grid, start, goal)
                if s_opt:
                    if rng.random() < 0.8:
                        all_train_maps.append(grid)
                        h_star_map = np.full((CONFIG["grid_size"], CONFIG["grid_size"]), np.inf)
                        h_star_map = compute_dijkstra_optimal_full(grid, goal) 
                        all_train_h_stars.append(h_star_map)
                    else:
                        test_pools[d].append((grid, c_opt, path_opt))
                    count += 1
        generation_stats[d]["attempts"] = attempts
        generation_stats[d]["reachable"] = count
        print(f"  Плотность {int(d*100)}%: {count} тестовых карт из {attempts} попыток")

    print("\n[2/4] Обучение CNN...")
    dataset = HeuristicDataset(all_train_maps, all_train_h_stars, CONFIG["patch_size"])
    dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
    cnn = LightweightCNN().to(CONFIG["device"])
    optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    cnn.train()
    for epoch in range(CONFIG["cnn_epochs"]):
        total_loss = 0
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(CONFIG["device"]), targets.to(CONFIG["device"])
            optimizer.zero_grad()
            loss = criterion(cnn(inputs).squeeze(), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{CONFIG['cnn_epochs']}, Loss: {total_loss/len(dataloader):.4f}")
    
    cnn.eval()
    torch.save(cnn.state_dict(), "output_data/cnn_weights.pth")
    
    print("\n[3/4] Прогрев CNN...")
    dummy_input = torch.randn(1, 2, CONFIG["patch_size"], CONFIG["patch_size"], device=CONFIG["device"])
    for _ in range(CONFIG["warmup_runs"]):
        with torch.no_grad():
            _ = cnn(dummy_input)

    print("\n[4/4] Профилированное тестирование...")
    all_records = []
    
    for d in CONFIG["densities"]:
        print(f"  Обработка плотности {int(d*100)}%...")
        for map_idx, (grid, cost_opt, path_opt) in enumerate(tqdm(test_pools[d], leave=False)):
            
            # Baseline
            stats_b = astar_profiled(grid, start, goal, cnn_model=None)
            # A*-LH
            stats_l = astar_profiled(grid, start, goal, cnn_model=cnn, patch_size=CONFIG["patch_size"])
            
            is_valid_lh = validate_path(grid, stats_l["path"], start, goal) if stats_l["success"] else False
            
            all_records.append({
                "density": d,
                "map_id": map_idx,
                "cost_opt": round(cost_opt, 4),
                "t_base_total_ms": round(stats_b["t_total_ms"], 4),
                "n_base_expanded": stats_b["n_expanded"],
                "t_lh_total_ms": round(stats_l["t_total_ms"], 4),
                "t_lh_prepare_ms": round(stats_l["t_prepare_ms"], 4),
                "t_lh_infer_ms": round(stats_l["t_infer_ms"], 4),
                "t_lh_queue_ms": round(stats_l["t_queue_ms"], 4),
                "n_lh_expanded": stats_l["n_expanded"],
                "n_lh_cnn_calls": stats_l["n_cnn_calls"],
                "n_lh_cache_hits": stats_l["n_cache_hits"],
                "cost_lh": round(stats_l["path_cost"], 4),
                "is_valid_lh": int(is_valid_lh)
            })
            
    csv_path = "output_data/per_map_results.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_records[0].keys())
        writer.writeheader()
        writer.writerows(all_records)
        
    with open("output_data/metadata.json", 'w') as f:
        json.dump({"config": CONFIG, "environment": get_env_info(), "generation_stats": generation_stats}, f, indent=2)
        
    print(f"\n Результаты сохранены в {csv_path}")
    return all_records, generation_stats

def compute_dijkstra_optimal_full(grid, goal):
    """Возвращает полную карту h_star для обучения"""
    size = grid.shape[0]
    h_star = np.full((size, size), np.inf)
    h_star[goal] = 0
    pq = [(0, goal)]
    while pq:
        cost, (x, y) = heapq.heappop(pq)
        if cost > h_star[x, y]: continue
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0: continue
                nx, ny = x + dx, y + dy
                if 0 <= nx < size and 0 <= ny < size and grid[nx, ny] == 0:
                    if dx != 0 and dy != 0:
                        if grid[x+dx, y] == 1 or grid[x, y+dy] == 1: continue
                    move_cost = 1.41421356 if (dx != 0 and dy != 0) else 1.0
                    new_cost = cost + move_cost
                    if new_cost < h_star[nx, ny]:
                        h_star[nx, ny] = new_cost
                        heapq.heappush(pq, (new_cost, (nx, ny)))
    return h_star

if __name__ == "__main__":
    records, gen_stats = run_experiment()
    
    # 6. АВТОМАТИЧЕСКАЯ ГЕНЕРАЦИЯ ТАБЛИЦ ИЗ ДАННЫХ
    print("\n" + "="*80)
    print("АВТОМАТИЧЕСКИ СГЕНЕРИРОВАННЫЕ ТАБЛИЦЫ (Single Source of Truth)")
    print("="*80)
    
    print("\n### Таблица 1. Статистика генерации карт")
    print("| Заданная плотность | Число попыток | Достижимых карт (Test 20%) | Фактическая плотность (оценка) |")
    print("| :---: | :---: | :---: | :---: |")
    for d in CONFIG["densities"]:
        print(f"| {int(d*100)}% | {gen_stats[d]['attempts']} | {gen_stats[d]['reachable']} | ~{int(d*100 * 1.5)}% (из-за дилатации) |")

    print("\n### Таблица 2. Вычисlительная эффективность и декомпозиция")
    print("| Плотность | N (карт) | Медиана T_LH / T_Base | T_prepare + T_infer (мс) | Доля overhead'а | Ср. время вызова CNN (мкс) |")
    print("| :---: | :---: | :---: | :---: | :---: | :---: |")
    for d in CONFIG["densities"]:
        subset = [r for r in records if r["density"] == d]
        n_maps = len(subset)
        if n_maps == 0: continue
        
        ratios_t = [r["t_lh_total_ms"] / max(r["t_base_total_ms"], 0.001) for r in subset]
        med_ratio = np.median(ratios_t)
        
        avg_prep = np.mean([r["t_lh_prepare_ms"] for r in subset])
        avg_inf = np.mean([r["t_lh_infer_ms"] for r in subset])
        avg_total_lh = np.mean([r["t_lh_total_ms"] for r in subset])
        avg_cnn_calls = np.mean([r["n_lh_cnn_calls"] for r in subset])
        
        overhead = avg_prep + avg_inf
        overhead_pct = (overhead / max(avg_total_lh, 0.001)) * 100
        avg_call_time = (overhead / max(avg_cnn_calls, 1)) * 1000
        
        print(f"| {int(d*100)}% | {n_maps} | {med_ratio:.2f} | {overhead:.2f} | {overhead_pct:.1f}% | {avg_call_time:.1f} |")

    print("\n### Таблица 3. Оценка оптимальности и валидности (на примере 30%)")
    d_target = 0.30
    subset_30 = [r for r in records if r["density"] == d_target]
    n_30 = len(subset_30)
    
    ratios_l = [r["cost_lh"] / max(r["cost_opt"], 0.001) for r in subset_30 if r["cost_opt"] > 0]
    med_l = np.median(ratios_l)
    max_l = np.max(ratios_l) if ratios_l else 1.0
    n_subopt = sum(1 for r in ratios_l if r > 1.0001)
    n_invalid = sum(1 for r in subset_30 if r["is_valid_lh"] == 0)
    
    print(f"| Метрика | Значение для плотности 30% (N={n_30}) |")
    print(f"| :--- | :--- |")
    print(f"| Медиана L_LH / L_Opt | {med_l:.4f} |")
    print(f"| Максимум L_LH / L_Opt | {max_l:.4f} |")
    print(f"| Число строго неоптимальных решений | {n_subopt} из {n_30} ({(n_subopt/n_30)*100:.1f}%) |")
    print(f"| Число невалидных маршрутов | {n_invalid} из {n_30} |")

 ЗАПУСК ВОСПРОИЗВОДИМОГО ЭКСПЕРИМЕНТА 

[1/4] Генерация карт...
  Плотность 10%: 100 тестовых карт из 150 попыток
  Плотность 20%: 100 тестовых карт из 457 попыток
  Плотность 30%: 71 тестовых карт из 2000 попыток
  Плотность 40%: 4 тестовых карт из 2000 попыток
  Плотность 50%: 0 тестовых карт из 2000 попыток

[2/4] Обучение CNN...
  Epoch 1/10, Loss: 157.8359
  Epoch 2/10, Loss: 69.2210
  Epoch 3/10, Loss: 48.3037
  Epoch 4/10, Loss: 44.5188
  Epoch 5/10, Loss: 43.7449
  Epoch 6/10, Loss: 43.0581
  Epoch 7/10, Loss: 42.4326
  Epoch 8/10, Loss: 42.0347
  Epoch 9/10, Loss: 41.6081
  Epoch 10/10, Loss: 40.9355

[3/4] Прогрев CNN...

[4/4] Профилированное тестирование...
  Обработка плотности 10%...


  Обработка плотности 20%...


  Обработка плотности 30%...


  Обработка плотности 40%...


  Обработка плотности 50%...



 Результаты сохранены в output_data/per_map_results.csv

АВТОМАТИЧЕСКИ СГЕНЕРИРОВАННЫЕ ТАБЛИЦЫ (Single Source of Truth)

### Таблица 1. Статистика генерации карт
| Заданная плотность | Число попыток | Достижимых карт (Test 20%) | Фактическая плотность (оценка) |
| :---: | :---: | :---: | :---: |
| 10% | 150 | 100 | ~15% (из-за дилатации) |
| 20% | 457 | 100 | ~30% (из-за дилатации) |
| 30% | 2000 | 71 | ~45% (из-за дилатации) |
| 40% | 2000 | 4 | ~60% (из-за дилатации) |
| 50% | 2000 | 0 | ~75% (из-за дилатации) |

### Таблица 2. Вычисlительная эффективность и декомпозиция
| Плотность | N (карт) | Медиана T_LH / T_Base | T_prepare + T_infer (мс) | Доля overhead'а | Ср. время вызова CNN (мкс) |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 10% | 20 | 53.49 | 75.93 | 95.4% | 144.6 |
| 20% | 21 | 46.46 | 58.63 | 95.6% | 140.9 |
| 30% | 14 | 43.90 | 44.82 | 96.1% | 148.0 |

### Таблица 3. Оценка оптимальности и валидности (на примере 30%)
| Метрика | Значение для плотности 30% (N=14